# 3. Taxonomy Classification
This notebook assigns taxonomic classifications using a custom-trained SILVA taxonomic classifier.

### Notebook Structure

**0.** Setup  
**1.** Train a Taxonomic classifier    
&nbsp;&nbsp;&nbsp;&nbsp;**1.1** Preparing the SILVA reference database   
&nbsp;&nbsp;&nbsp;&nbsp;**1.2** Training an amplicon-region specific classifier   
**2.** Taxonomy assignment  
**3.** Evaluate taxonomic classifier   
**4.** Taxonomic classifier comparison

### 0. Setup

In [1]:
# Import all necessary packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2

%matplotlib inline

In [2]:
# Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# Data directories
raw_data_dir = "../data/raw"
meta_data_dir = "../data/processed/metadata"
denoised_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
# Create the new taxonomy folder

In [4]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

## 1. Train a Taxonomic classifier
We train a custom Silva classifier on the V4 region of the 16S rRNA gene based on these two QIIME tutorials:
- [Processing, filtering, and evaluating the SILVA database (and other reference sequence data) with RESCRIPt](https://forum.qiime2.org/t/processing-filtering-and-evaluating-the-silva-database-and-other-reference-sequence-data-with-rescript/15494)
- [Using RESCRIPt's 'extract-seq-segments' to extract reference sequences without PCR primer pairs](https://forum.qiime2.org/t/using-rescripts-extract-seq-segments-to-extract-reference-sequences-without-pcr-primer-pairs/23618)

To reduce computation time and avoid memory issues on JupyterHub, the classifier was trained on Euler. 
The script is available in the *scripts/additional* folder under the name *'3_euler_script_train_region_specific_classifier.sh'*. The conda environment configuration file for Euler can be found in the environment folder: *environment/euler_env1.yml*.

All code snippets are also included in this notebook and the first part (1.1 Preparing the SILVA reference database) runs on JupyterHub.

The resulting classifier was uploaded to Polybox and then downloaded here, as the file was too large for GitHub.

### 1.1 Preparing the SILVA reference database 
Pipeline for Preparing the SILVA reference database and then training the classifier:
1. Download Silva reference (RNA) and reverse transcribe into DNA
2. Filter out poor quality sequences 
3. Filter sequences by length and taxonomy
4. Dereplicate sequences
5. Extract amplicon-specific region for classifier
6. Dereplicate again after trimming

In the next two cells we are downloading the **SILVA 138.2 SSURef NR99 RNA reference** database via QIIME RESCRIPt.

**SILVA 138.2** is a curated, high-quality database of ribosomal RNA sequences frequently used for 16S/18S taxonomic classification.

**SSU** → Small Subunit rRNA (in our case 16S rRNA)  
**Ref** → Refined   
**NR99** → Non-redundant at 99% identity (duplicate sequences are clustered so no two sequences are more than 99% identical).   

In [5]:
# Download Silva reference (RNA) --> missing permission on euler to download the data directly like this
! qiime rescript get-silva-data \
    --p-version '138.2' \
    --p-target 'SSURef_NR99' \
    --o-silva-sequences  $taxonomy_data_dir/silva-138.2-ssu-nr99-rna-seqs.qza \
    --o-silva-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[RNASequence] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-rna-seqs.qza
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/silva-138.2-ssu-nr99-tax.qza


In [ ]:
# Reverse transcribe the RNA into DNA
! qiime rescript reverse-transcribe \
    --i-rna-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-rna-seqs.qza \
    --o-dna-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs.qza

In the next two cells we filter out sequences with poor quality. Qiimes rescript cull-seqs filters sequences with more than 4 ambiguous bases (N’s, R, Y, etc.), sequences with homopolymers longer than 7 bp, sequences containing invalid characters and sequences with extremely short or degenerate content. As a next step we also filter the sequences by length to remove partial and abnormal sequences and taxonomy to remove invalid, missing, ambiguous, or irrelevant taxonomy.

In [ ]:
# Filter out poor quality 
! qiime rescript cull-seqs \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs.qza \
    --o-clean-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-cleaned.qza

In [ ]:
# Filtering sequences by length and taxonomy
! qiime rescript filter-seqs-length-by-taxon \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-cleaned.qza \
    --i-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza \
    --p-labels Archaea Bacteria \
    --p-min-lens 900 1200 \
    --o-filtered-seqs $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-filt.qza \
    --o-discarded-seqs $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-discard.qza

In the next cells we dereplicate our sequences to ensure that each unique sequence only appears once. These unique reads are trimmed to only contain the V4 region, so that we can use it for our classifier. Since trimming can create duplicates again, we perform dereplication a second time.

In [ ]:
# Dereplicate
! qiime rescript dereplicate \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-filt.qza  \
    --i-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax.qza \
    --p-mode 'uniq' \
    --o-dereplicated-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-derep-uniq.qza \
    --o-dereplicated-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-derep-uniq.qza

In [ ]:
# Make amplicon-region specific classifier
! qiime feature-classifier extract-reads \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-derep-uniq.qza \
    --p-f-primer GTGYCAGCMGCCGCGGTAA \
    --p-r-primer GGACTACNVGGGTWTCTAAT \
    --p-n-jobs 2 \
    --p-read-orientation 'forward' \
    --o-reads $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r.qza

In [ ]:
# Dereplicate again (could have new replicates in the shorter regions)
! qiime rescript dereplicate \
    --i-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r.qza \
    --i-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-derep-uniq.qza \
    --p-mode 'uniq' \
    --o-dereplicated-sequences $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza \
    --o-dereplicated-taxa $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza

### 1.2 Training an amplicon-region specific classifier

Now we need to train our amplicon-region specific classifier. There is not enough RAM available to run this in the notebook so we trained the classifier on euler. We added a wget to import the trained classifier. The euler scripts are available in the "additional" folder. The classifier is downloaded from Julia's Polybox into the taxonomy folder.

In [ ]:
# Train amplicon-region specific classifier 
"""
! qiime feature-classifier fit-classifier-naive-bayes \
    --i-reference-reads $taxonomy_data_dir/silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza \
    --i-reference-taxonomy $taxonomy_data_dir/silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza \
    --p-verbose \
    --o-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-515f-806r-classifier.qza
"""

In [6]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

# Download the trained classifier
wget --progress=bar:force:noscroll \
  -O "$1/silva-138.2-ssu-nr99-515f-806r-classifier.qza" \
  "https://polybox.ethz.ch/index.php/s/crQxtTa7MHXEKAk/download"

--2025-12-10 17:07:30--  https://polybox.ethz.ch/index.php/s/crQxtTa7MHXEKAk/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63373448 (60M) [application/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-classifier.qza’

../data/processed/t 100%[===================>]  60.44M   118MB/s    in 0.5s    

2025-12-10 17:07:31 (118 MB/s) - ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-classifier.qza’ saved [63373448/63373448]



## 2. Taxonomy assignment

As a next step we can use our trained classifier to assign taxonomy labels to our ASVs. And create some visualizations.

In [7]:
# Assign taxonomy labels to our ASVs 
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-515f-806r-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy.qza


In [8]:
# Check if it created the taxonomy artefact
! qiime tools peek $taxonomy_data_dir/taxonomy.qza

UUID:        aaaeff6c-6ee7-499e-840f-52a2d43a7df2
Type:        FeatureData[Taxonomy]
Data format: TSVTaxonomyDirectoryFormat


In [9]:
# Create the visualization
! qiime metadata tabulate \
    --m-input-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $taxonomy_data_dir/taxonomy.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxonomy.qzv


In [10]:
Visualization.load(f"{taxonomy_data_dir}/taxonomy.qzv")

<visualization: Visualization uuid: 31fb63a4-f3cc-4c41-8715-4eec301dc61a>

In [11]:
# Create interactive taxonomy bar plot
! qiime taxa barplot \
    --i-table $denoised_data_dir/dada2_table.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --m-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $taxonomy_data_dir/taxa-bar-plots.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxa-bar-plots.qzv


In [12]:
Visualization.load(f"{taxonomy_data_dir}/taxa-bar-plots.qzv")

<visualization: Visualization uuid: a4b7a5c5-1247-4813-b549-93a5b3ccbf63>

We can see that there are sequences from eukaryotes, mitochondria and chloroplasts in our data. So as a next step we will filter them out and have a look at the barplot again.

In [13]:
! qiime taxa filter-table \
    --i-table $denoised_data_dir/dada2_table.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --p-exclude Mitochondria,Chloroplast,Eukaryota \
    --o-filtered-table $taxonomy_data_dir/dada2_table_filtered.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureTable[Frequency] to: ../data/processed/taxonomy/dada2_table_filtered.qza


In [14]:
! qiime taxa filter-seqs \
    --i-sequences $denoised_data_dir/dada2_rep_seq.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --p-exclude Mitochondria,Chloroplast,Eukaryota \
    --o-filtered-sequences $taxonomy_data_dir/dada2_rep_seq_filtered.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Sequence] to: ../data/processed/taxonomy/dada2_rep_seq_filtered.qza


In [15]:
# Create interactive taxonomy bar plot again
! qiime taxa barplot \
    --i-table $taxonomy_data_dir/asv_table_filtered.qza \
    --i-taxonomy $taxonomy_data_dir/taxonomy.qza \
    --m-metadata-file $meta_data_dir/metadata_merged.tsv \
    --o-visualization $taxonomy_data_dir/taxa-bar-plots-filtered.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxa-bar-plots-filtered.qzv


In [16]:
Visualization.load(f"{taxonomy_data_dir}/taxa-bar-plots-filtered.qzv")

<visualization: Visualization uuid: 4517856e-adc0-409f-89bc-5b3f8d8253e6>

In the barplots we don't see any eukaryotes so the filtering was successful.

Now we can quickly check how many sequences were identified up to which level.

In [17]:
# Load QIIME 2 artifact files as python objects
taxa = q2.Artifact.load(f'{taxonomy_data_dir}/taxonomy.qza')
# view as a `pandas.DataFrame`. Note: Only some Artifact types can be transformed to DataFrames
taxa = taxa.view(pd.DataFrame)

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [18]:
# Count for each taxonomic level how many ASVs were still identified
ranks = [("d__", "Domain"), ("p__", "Phylum"), ("c__", "Class"), ("o__", "Order"), ("f__", "Family"), ("g__", "Genus"), ("s__", "Species")]
total = len(taxa)
for i, (prefix, name) in enumerate(ranks, start=1):
    count = taxa["Taxon"].str.contains(prefix).sum()
    percent = count / total * 100
    print(f"{i}. {name}: {count} ({percent:.1f}%)")

1. Domain: 3873 (100.0%)
2. Phylum: 3870 (99.9%)
3. Class: 3868 (99.9%)
4. Order: 3855 (99.5%)
5. Family: 3790 (97.9%)
6. Genus: 3341 (86.3%)
7. Species: 3341 (86.3%)


Nearly all sequences were successfully classified, from 100% at the domain level to 86.3% at the genus and species levels. The identical genus and species counts are expected, since SILVA treats species identifications as extensions of the genus level, marked with the s__ prefix.

## 3. Evaluate taxonomic classifier
There are multiple ways to evaluate a taxonomic classifier. One option is **evaluate-cross-validate**, which performs k-fold cross-validation by splitting the database into K subsets and testing each fold using a classifier trained on the remaining K-1 folds. We do not use this method because it is very time-consuming for large databases.

Instead, we use **evaluate-fit-classifier**, which trains and tests a classifier on the entire database, creating a "perfect classifier" with intentional data leakage. This simulates the best possible classification accuracy, assuming all query sequences have a perfect match in the reference database. 
We chose this method because it is much quicker. However, it is important to note that this evaluation provides only an upper bound and real-world performance may be lower when classifying sequences not present in the reference database.

In [ ]:
# Evaluate classifier (done on euler)
"""
! qiime rescript evaluate-fit-classifier \
  --i-sequences silva-138.2-ssu-nr99-seqs-515f-806r-uniq.qza \
  --i-taxonomy silva-138.2-ssu-nr99-tax-515f-806r-derep-uniq.qza \
  --o-classifier silva-138.2-ssu-nr99-515f-806r-fit-classifier.qza \
  --o-observed-taxonomy silva-138.2-ssu-nr99-515f-806r-predicted-taxonomy.qza \
  --o-evaluation silva-138.2-ssu-nr99-515f-806r-fit-classifier-evaluation.qzv
"""

In [19]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

# Download the QIIME2 Visualization for the taxonomy evaluation 
wget --progress=bar:force:noscroll \
  -O "$1/silva-138.2-ssu-nr99-515f-806r-fit-classifier-evaluation.qzv" \
  "https://polybox.ethz.ch/index.php/s/9njXwCnb6HbK6Gz/download"

--2025-12-10 17:11:53--  https://polybox.ethz.ch/index.php/s/9njXwCnb6HbK6Gz/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 386135 (377K) [application/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-fit-classifier-evaluation.qzv’

../data/processed/t 100%[===================>] 377.08K  --.-KB/s    in 0.03s   

2025-12-10 17:11:53 (14.7 MB/s) - ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-515f-806r-fit-classifier-evaluation.qzv’ saved [386135/386135]



In [20]:
Visualization.load(f"{taxonomy_data_dir}/silva-138.2-ssu-nr99-515f-806r-fit-classifier-evaluation.qzv")

<visualization: Visualization uuid: 025010c9-eafc-45fd-8b4a-6703ac68ff0b>

At the first taxonomic level, the upper bound for performance remains at 100% for all metrics. However, accuracy begins to decrease at subsequent levels and by the last level, recall drops below 90% (87%). Precision and F-score remain above 90% at the final level, so they are still quite good. However, while SILVA is well-curated for human stool 16S rRNA sequences, not all of our sequences may be represented, so the true performance could be even lower.

## 4. Taxonomic classifier comparison
We have already trained two custom classifiers and will download a pretrained one from the internet to compare their performance. We want to test how many sequences each classifier can identify at different taxonomic levels.

The classifiers:

- **Region-specific:** A custom SILVA classifier trained on the V4 region of the 16S rRNA gene. (Code in this notebook & euler script for the last step in the 'additional' folder (in scripts) under the name '3_euler_script_train_region_specific_classifier.sh')

- **Full-length:** Identical to the region-specific classifier, but trained on the entire SILVA database rather than primer-trimmed sequences. This training was substantially more time-intensive, taking over 20 hours on Euler. (Euler script available in the scripts/additional folder under the name '3_euler_script_train_full_length_classifier.sh')

- **Pretrained classifier:** This classifier is downloaded from the [QIIME 2 data resources page](https://library.qiime2.org/data-resources). We chose a trained model weighted according to environment-specific (human stool) taxonomic abundance information, since this can significantly increase species-level classification accuracy across common sample types [[Kaehler, 2019]](https://www.nature.com/articles/s41467-019-12669-6). This classifier is also based on the SILVA reference database.
    
First, we need to download the pretrained classifier and generate taxonomic assignments for it.

In [21]:
# Dowload the pretrained, weighted – SILVA classifier
! wget -O $taxonomy_data_dir/silva-138-99-nb-human-stool-weighted-classifier.qza \
https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza

--2025-12-10 17:11:53--  https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza [following]
--2025-12-10 17:11:54--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-human-stool-weighted-classifier.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.92.185.152, 52.92.136.248, 52.92.128.40, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.92.185.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218311668 (208M) [binary/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138-99-nb-h

In [22]:
# Predict the taxonomy with the pretrained silva classifier
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138-99-nb-human-stool-weighted-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy_pretrained_silva.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy_pretrained_silva.qza


Then we have to make sure we also have access to the full-length self-trained taxonomic classifier. We trained it on Euler and added it to the notebook via Polybox.

In [23]:
%%bash -s "$taxonomy_data_dir"
mkdir -p "$1"

# Training the full-length taxonomy classifier took more than 20 hours on Euler,
# so the script was added to the additional folder, and the classifier was uploaded to Polybox and downloaded here.
wget --progress=bar:force:noscroll \
  -O "$1/silva-138.2-ssu-nr99-classifier.qza" \
  "https://polybox.ethz.ch/index.php/s/oq4wjw2JKpqbwYb/download"

--2025-12-10 17:13:12--  https://polybox.ethz.ch/index.php/s/oq4wjw2JKpqbwYb/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221286863 (211M) [application/octet-stream]
Saving to: ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-classifier.qza’

../data/processed/t 100%[===================>] 211.04M   289MB/s    in 0.7s    

2025-12-10 17:13:13 (289 MB/s) - ‘../data/processed/taxonomy/silva-138.2-ssu-nr99-classifier.qza’ saved [221286863/221286863]



In [24]:
# Predict the taxonomy with the selftrained, full-length silva classifier
! qiime feature-classifier classify-sklearn \
    --i-classifier $taxonomy_data_dir/silva-138.2-ssu-nr99-classifier.qza \
    --i-reads $denoised_data_dir/dada2_rep_seq.qza \
    --o-classification $taxonomy_data_dir/taxonomy_custom_full_length.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[Taxonomy] to: ../data/processed/taxonomy/taxonomy_custom_full_length.qza


Next, we assess the taxonomic information content of each reference, including label counts, entropy, and rank distribution. This allows us to quantify differences in completeness and richness for the region-specific, full-length, and pretrained classifiers.

In [25]:
! qiime rescript evaluate-taxonomy \
  --i-taxonomies $taxonomy_data_dir/taxonomy.qza $taxonomy_data_dir/taxonomy_custom_full_length.qza $taxonomy_data_dir/taxonomy_pretrained_silva.qza\
  --p-labels region_specific full_length pretrained\
  --o-taxonomy-stats $taxonomy_data_dir/taxonomy-evaluation.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/taxonomy/taxonomy-evaluation.qzv


In [26]:
Visualization.load(f"{taxonomy_data_dir}/taxonomy-evaluation.qzv")

<visualization: Visualization uuid: 6574988f-5fd7-4c1b-846a-f019ab20edee>

The custom SILVA classifier trained on the V4 region of the 16S rRNA gene assigns more sequences to the deepest taxonomic level. This is likely because trimming the reference sequences to match the V4 region better tunes the classifier to the query sequences, reducing ambiguity with unrelated sequences. Since it also exhibits the highest taxonomic entropy, we choose to use the region-specific classifications in the subsequent notebooks.

# References
Kaehler, B.D., Bokulich, N.A., McDonald, D. et al. Species abundance information improves sequence taxonomy classification accuracy. Nat Commun 10, 4643 (2019). https://doi.org/10.1038/s41467-019-12669-6

Robeson, M. (2020, Juni 25). Processing, filtering, and evaluating the SILVA database (and other reference sequence data) with RESCRIPt—Community Contributions / Tutorials. QIIME 2 Forum. https://forum.qiime2.org/t/processing-filtering-and-evaluating-the-silva-database-and-other-reference-sequence-data-with-rescript/15494

Robeson, M. (2022, Juli 13). Using RESCRIPt’s „extract-seq-segments“ to extract reference sequences without PCR primer pairs. - Community Contributions / Tutorials. QIIME 2 Forum. https://forum.qiime2.org/t/using-rescripts-extract-seq-segments-to-extract-reference-sequences-without-pcr-primer-pairs/23618


<span style="color:gray">

**Citations for the tutorials:**

Michael S Robeson II, Devon R O'Rourke, Benjamin D Kaehler, Michal Ziemski, Matthew R Dillon, Jeffrey T Foster, Nicholas A Bokulich. 2021. "RESCRIPt: Reproducible sequence taxonomy reference database management". PLoS Computational Biology 17 (11): e1009581.; doi: 10.1371/journal.pcbi.1009581

Pruesse, Elmar, Christian Quast, Katrin Knittel, Bernhard M. Fuchs, Wolfgang Ludwig, Jörg Peplies, and Frank Oliver Glöckner. 2007. “SILVA: A Comprehensive Online Resource for Quality Checked and Aligned Ribosomal RNA Sequence Data Compatible with ARB.” Nucleic Acids Research 35 (21): 7188–96. doi: 10.1093/nar/gkm864

Quast, Christian, Elmar Pruesse, Pelin Yilmaz, Jan Gerken, Timmy Schweer, Pablo Yarza, Jörg Peplies, and Frank Oliver Glöckner. 2013. “The SILVA Ribosomal RNA Gene Database Project: Improved Data Processing and Web-Based Tools.” Nucleic Acids Research 41: D590–96. doi: 10.1093/nar/gks1219

Torbjørn Rognes, Tomáš Flouri, Ben Nichols, Christopher Quince, and Frédéric Mahé. 2016. “VSEARCH: A Versatile Open Source Tool for Metagenomics.” PeerJ 4 (October): e2584. doi: 10.7717/peerj.2584

**Citations for the pretrained classifier:**

Robeson, M. S., II, O’Rourke, D. R., Kaehler, B. D., Ziemski, M., Dillon, M. R., Foster, J. T., & Bokulich, N. A. (2020). RESCRIPt: Reproducible sequence taxonomy reference database management for the masses. In bioRxiv. bioRxiv.

Bokulich, N. A., Kaehler, B. D., Rideout, J. R., Dillon, M., Bolyen, E., Knight, R., Huttley, G. A., & Gregory Caporaso, J. (2018). Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2’s q2-feature-classifier plugin. Microbiome, 6(1).

Kaehler, B. D., Bokulich, N. A., McDonald, D., Knight, R., Caporaso, J. G., & Huttley, G. A. (2019). Species abundance information improves sequence taxonomy classification accuracy. Nat. Commun., 10(1), 4643.
</span>